In [8]:
import os
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score,
)
from xgboost import XGBRegressor

# ==========================
# Load Dataset
# ==========================
path = Path("G:/infotact_2/notebook/clean_house_data.csv")
data = pd.read_csv(path)

# Convert date column
try:
    data["date"] = pd.to_datetime(
        data["date"],
        format="%Y%m%dT%H%M%S",
        errors="coerce"
    )
except ValueError:
    data["date"] = pd.to_datetime(data["date"], errors="coerce")

# ==========================
# Feature Engineering
# ==========================
feature_cols = [
    "bedrooms",
    "bathrooms",
    "sqft_living",
    "sqft_lot",
    "floors",
    "waterfront",
    "view",
    "condition",
    "grade",
    "sqft_above",
    "sqft_basement",
    "yr_built",
    "yr_renovated",
    "zipcode",
    "lat",
    "long",
    "sqft_living15",
    "sqft_lot15",
    "house_age",
    "renovated",
]

X = data[feature_cols].copy()

X["sale_year"] = data["date"].dt.year
X["sale_month"] = data["date"].dt.month

# One-hot encode zipcode
X = pd.get_dummies(X, columns=["zipcode"], drop_first=True)

y = data["price"]

print("Training Shape:", X.shape)

# ==========================
# Train-Test Split
# ==========================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# ==========================
# Train XGBoost Model
# ==========================
model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42,
    n_jobs=-1,
)

model.fit(X_train, y_train)

# ==========================
# Evaluate Model
# ==========================
pred = model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
mape = mean_absolute_percentage_error(y_test, pred)
r2 = r2_score(y_test, pred)

print(f"MAE  : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"MAPE : {mape:.4f}")
print(f"R²   : {r2:.4f}")

print("\nSample Predictions:")
print(np.round(pred[:10], 2))

print("\nActual Values:")
print(np.round(y_test.iloc[:10].to_numpy(), 2))

# ==========================
# Save Model
# ==========================
os.makedirs("../model", exist_ok=True)

joblib.dump(model, "../model/house_price_model.pkl")

# Save feature names (important for Streamlit)
joblib.dump(X.columns.tolist(), "../model/model_columns.pkl")

print("\n✅ Model saved successfully!")
print("📁 Location: model/house_price_model.pkl")

print("\n✅ Feature columns saved successfully!")
print("📁 Location: model/model_columns.pkl")

Training Shape: (21613, 90)
MAE  : 65936.79
RMSE : 134160.92
MAPE : 0.1200
R²   : 0.8809

Sample Predictions:
[3.8107538e+05 8.5566094e+05 1.0838544e+06 1.9838695e+06 7.1287438e+05
 2.4096259e+05 7.7598394e+05 6.4499938e+05 4.5169819e+05 6.0723444e+05]

Actual Values:
[ 365000.  865000. 1038000. 1490000.  711000.  211000.  790000.  680000.
  384500.  605000.]

✅ Model saved successfully!
📁 Location: model/house_price_model.pkl

✅ Feature columns saved successfully!
📁 Location: model/model_columns.pkl
